# Model 1 baseline on Kaggle

Run these cells in your configured GPU notebook with the writable `m3-kaggle` checkout and G1 meshes. The compatibility cell applies the included M1 source update when needed. This run uses 8 environments, seed 42, and 1.6 million steps to match the previous M3 experiment. The M1 actor receives the current observation. Training and checkpoint verification run in separate processes.

Set `reference` to the existing M3 result folder, including its settings and both evaluation JSON files. A saved Kaggle dataset path also works.

In [ ]:
import json
import os
from pathlib import Path
import statistics
import subprocess
import sys

repo = Path("/kaggle/working/Bipedal-Locomotion")
run = repo / "results/m1_1p6m_env8_lr1e4"
reference = repo / "results/m3_1p6m_env8_lr1e4"
env = {**os.environ, "XLA_PYTHON_CLIENT_PREALLOCATE": "false"}

def read(folder, filename):
    return json.loads((folder / filename).read_text())

if not (repo / "scripts/train_kaggle.py").is_file():
    raise FileNotFoundError(f"Point repo to the writable training checkout: {repo}")

## Enable M1 in the checkout

The update targets the training entry point, observation builder, and PPO trainer. A conflicting edit stops the update before training starts.

In [ ]:
patch = r'''diff --git a/scripts/train_kaggle.py b/scripts/train_kaggle.py
index 17d1537..7619bb2 100644
--- a/scripts/train_kaggle.py
+++ b/scripts/train_kaggle.py
@@ -31,33 +31,40 @@ def main():
                         help="Path to terrain configuration YAML")
     parser.add_argument("--num-envs", type=int, default=None,
                         help="Total parallel environments (sharded across GPUs)")
-    parser.add_argument("--history-len", type=int, default=5,
-                        help="Number of history timesteps (k)")
+    parser.add_argument("--history-len", type=int,
+                        help="Number of history timesteps (M1: 0, other models default to 5)")
     parser.add_argument("--seed", type=int, default=42,
                         help="Master PRNG seed")
     parser.add_argument("--output-dir", type=str, default=None,
                         help="Checkpoint and logs output directory")
-    parser.add_argument("--num-timesteps", type=int, help="Exact M3 environment step budget")
-    parser.add_argument("--smoke", action="store_true", help="Run two M3 optimizer updates (pilot default: ten)")
+    parser.add_argument("--num-timesteps", type=int, help="Exact M1 or M3 environment step budget")
+    parser.add_argument("--smoke", action="store_true", help="Run two PPO optimizer updates (pilot default: ten)")
     args = parser.parse_args()
-    if args.model == "M3":
+    if args.history_len is None:
+        args.history_len = 0 if args.model == "M1" else 5
+    if args.model in ("M1", "M3"):
+        if args.model == "M1" and args.history_len != 0:
+            parser.error("M1 uses only the current observation (--history-len 0)")
+        if args.model == "M3" and args.history_len < 1:
+            parser.error("M3 history length must be positive")
         if args.category is not None or args.difficulty != 0 or Path(args.terrain_config).resolve() != ROOT / "configs/terrain.yaml":
-            parser.error("M3 uses the established terrain configuration and curriculum")
-        if args.history_len < 1 or any(v is not None and v <= 0 for v in (args.num_envs, args.num_timesteps)):
-            parser.error("History length, environment count, and step budget must be positive")
+            parser.error(f"{args.model} uses the established terrain configuration and curriculum")
+        if any(v is not None and v <= 0 for v in (args.num_envs, args.num_timesteps)):
+            parser.error("Environment count and step budget must be positive")
         if args.smoke and args.num_timesteps is not None:
             parser.error("Choose either --smoke or an explicit --num-timesteps")
-        args.output_dir = args.output_dir or ("results/m3_smoke" if args.smoke else "results/m3_pilot")
+        kind = "smoke" if args.smoke else "pilot"
+        args.output_dir = args.output_dir or f"results/{args.model.lower()}_{kind}"
         try:
             from src.training.m3_pilot import run
         except ImportError as exc:
-            parser.exit(2, f"M3 needs the configured Playground/Brax runtime: {exc}\n")
+            parser.exit(2, f"{args.model} needs the configured Playground/Brax runtime: {exc}\n")
         run(args)
         # Release the trainer's GPU allocations before verifying in a fresh interpreter.
         os.execv(sys.executable, [sys.executable, "-m", "src.training.m3_pilot", args.output_dir])
 
     if args.num_timesteps is not None or args.smoke:
-        parser.error("--num-timesteps and --smoke apply to M3")
+        parser.error("--num-timesteps and --smoke apply to M1 and M3")
     os.environ.update(XLA_PYTHON_CLIENT_PREALLOCATE="false",
                       XLA_PYTHON_CLIENT_MEM_FRACTION="0.85", TF_FORCE_GPU_ALLOW_GROWTH="true")
     args.num_envs = args.num_envs or 4096
diff --git a/src/training/m3_history.py b/src/training/m3_history.py
index 03ad757..5120aab 100644
--- a/src/training/m3_history.py
+++ b/src/training/m3_history.py
@@ -2,8 +2,10 @@
 
 
 def initial_input(obs, action_width, length, xp):
-    if length < 1:
-        raise ValueError("history length must be positive")
+    if length == 0:
+        return obs
+    if length < 0:
+        raise ValueError("history length must be nonnegative")
     shape = (*obs.shape[:-1], length * (obs.shape[-1] + action_width))
     return xp.concatenate((obs, xp.zeros(shape, dtype=obs.dtype)), axis=-1)
 
@@ -11,6 +13,8 @@ def initial_input(obs, action_width, length, xp):
 def next_input(previous, action, observation, done, xp):
     """Advance one transition. On autoreset, observation is the new reset observation."""
     width = observation.shape[-1]
+    if previous.shape[-1] == width:
+        return observation
     pair_width = width + action.shape[-1]
     history = xp.concatenate(
         (previous[..., width + pair_width:], previous[..., :width], action), axis=-1
diff --git a/src/training/m3_pilot.py b/src/training/m3_pilot.py
index 256e9f4..0881c33 100644
--- a/src/training/m3_pilot.py
+++ b/src/training/m3_pilot.py
@@ -1,4 +1,4 @@
-"""Bounded M3 PPO training with matched initial/final evaluations."""
+"""Bounded M1 and M3 PPO training with matched initial/final evaluations."""
 
 from functools import partial
 from hashlib import sha256
@@ -65,7 +65,7 @@ def evaluate(env, policy, horizon):
             values = np.asarray([state.reward, state.metrics["m3_distance"],
                                  state.metrics["m3_tracking"], state.done])
             if not np.isfinite(values).all():
-                raise ValueError("Non-finite M3 evaluation transition")
+                raise ValueError("Non-finite evaluation transition")
             reward += float(values[0])
             distance += float(values[1])
             error += float(values[2])
@@ -177,7 +177,7 @@ def run(args):
     width = actor_shape.shape[-1]
     expected = width + args.history_len * (width + env.action_size)
     if observation_shapes["state"] != (expected,):
-        raise ValueError("Simulator observation shape disagrees with the M3 history contract")
+        raise ValueError("Simulator observation shape disagrees with the configured actor width")
 
     ppo = locomotion_params.brax_ppo_config(TASK).to_dict()
     ppo["learning_rate"] = 0.0001
@@ -189,7 +189,8 @@ def run(args):
     ppo.update(num_envs=nenv, batch_size=nenv, num_minibatches=1, num_updates_per_batch=1,
                num_timesteps=budget, num_evals=1, num_eval_envs=nenv, num_resets_per_eval=0,
                run_evals=False)
-    settings = dict(model="M3", architecture="PPO MLP with observation–action history stacking",
+    settings = dict(model=args.model, architecture=("PPO MLP with current observation" if args.model == "M1"
+                                                 else "PPO MLP with observation–action history stacking"),
         task=TASK, seed=args.seed, actor_fields=[
         "pelvis gyro", "projected gravity", "velocity command", "joint offsets", "joint velocities"],
         observation_width=width, actions=env.action_size, actor_width=expected,
@@ -201,7 +202,7 @@ def run(args):
         observation_shapes=observation_shapes, optimizer_updates=budget // unit,
         terrain_config=str(terrain_path.resolve()), output_dir=str(out),
         devices=[f"{d}: {d.device_kind}" for d in jax.local_devices()])
-    print("Resolved M3 run configuration", flush=True)
+    print(f"Resolved {args.model} run configuration", flush=True)
     print(json.dumps(settings, indent=2, allow_nan=False), flush=True)
     save_json(out / "settings.json", settings)
 
'''
def check_patch(*flags):
    return subprocess.run(["git", "apply", *flags, "-"], cwd=repo, input=patch,
                          text=True, capture_output=True)

if check_patch("--reverse", "--check").returncode != 0:
    result = check_patch("--check")
    if result.returncode:
        raise RuntimeError("Use the updated m3-kaggle checkout. " + result.stderr)
    check_patch().check_returncode()
print("M1 trainer ready.")

## Train and verify the baseline

A completed run goes directly to checkpoint verification. A partial run needs a new output folder for a fresh start.

In [ ]:
if (run / "training.json").is_file():
    command = [sys.executable, "-u", "-m", "src.training.m3_pilot", str(run)]
else:
    if run.exists() and any(run.iterdir()):
        raise FileExistsError(f"Partial run at {run}. Choose an empty output directory.")
    command = [sys.executable, "-u", "scripts/train_kaggle.py", "--model", "M1",
               "--num-envs", "8", "--num-timesteps", "1600000", "--seed", "42",
               "--output-dir", str(run)]
subprocess.run(command, cwd=repo, env=env, check=True)
print(read(run, "reload.json"))

## Compare on the same seeds

Longer episodes and lower tracking error indicate progress. Early terminations include torso inversion and leg collisions.

In [ ]:
m1 = read(run, "settings.json")
m3 = read(reference, "settings.json")
if (m1["model"], m1["history_length"], m3["model"], m3["history_length"]) != ("M1", 0, "M3", 5):
    raise ValueError("Select the M1 baseline and five-step M3 reference runs.")
for key in ("task", "seed", "terrain_sha256", "terrain", "difficulty", "ppo", "network",
            "observation_width", "actions", "control_dt", "physics_dt", "simulator_backend"):
    if m1[key] != m3[key]:
        raise ValueError(f"Run settings differ for {key}")

seeds = None
print("Policy / stage       Duration    Tracking error    Return    Early ends")
for folder, settings in ((run, m1), (reference, m3)):
    for stage in ("initial", "trained"):
        episodes = read(folder, f"{stage}_evaluation.json")["episodes"]
        ids = sorted(e["seed"] for e in episodes)
        if len(set(ids)) != 10 or len(ids) != 10 or (seeds is not None and ids != seeds):
            raise ValueError("Expected ten matched evaluation seeds.")
        seeds = ids
        duration = statistics.mean(e["steps"] for e in episodes) * settings["control_dt"]
        error = statistics.mean(e["tracking_error_mps"] for e in episodes)
        reward = statistics.mean(e["episode_return"] for e in episodes)
        ended = sum(e["fall"] for e in episodes)
        print(f'{settings["model"]} / {stage:<8}      {duration:.3f} s       {error:.3f} m/s'
              f'     {reward:7.3f}      {ended}/10')